# Phase 5: Gaussian Wells + SARF Anchors — OpenWebText Scale-Up

## Motivation

Phase 4 demonstrated that the SQ3 structured V_θ suffers from **persistent
training instabilities** on OpenWebText due to its unbounded potential form.
Three rounds of containment fixes (log1p penalty, LR/clip tuning, EMA watchdog)
managed to reach 234 PPL but exhibited a "doom loop" of repeated watchdog
triggers that prevented sustained progress.

This Phase 5 notebook scales the **Gaussian mixture-PDF** V_θ from the
TinyStories ablation (`colab_fock_gaussian_sarf_vtheta.ipynb`) to OpenWebText.
The Gaussian form provides structurally bounded V ∈ [-Σw_k, 0] and bounded
forces, eliminating the root cause of all three SQ3 blowup modes.

## V_θ variant selection

Run the TinyStories ablation first. Then set `V_THETA_VARIANT` below to the
best-performing variant (expected: `gaussian` or `sarf`).

## Config summary

| Parameter | Phase 4 (SQ3) | **Phase 5 (Gaussian/SARF)** |
|-----------|---------------|-----------------------------|
| V_θ form | SQ3 log-sum-exp (unbounded) | Gaussian PDF (bounded) |
| V range | (-∞, +∞) | [-Σw_k, 0] |
| d | 384 | 384 |
| L | 16 | 16 |
| K_mix / N_S | 8 | 8 / 64 (config dependent) |
| Total steps | 200,000 | 200,000 |
| Watchdog | threshold=10.0, patience=100 | threshold=20.0, patience=200 |
| LR | 1.2e-4 | 2e-4 (bounded V allows higher LR) |

## Expected benefits over Phase 4

1. **No Blowup 1** — V² penalty is bounded by construction
2. **Reduced Blowup 2/3** — force magnitude capped at 0.607 w/σ
3. **No doom loop** — watchdog triggers should be rare or absent
4. **Higher LR feasible** — bounded dynamics tolerate faster learning

## Prerequisites

- TinyStories ablation completed (know which variant wins)
- H100 80 GB GPU (same as Phase 4)
- OpenWebText tokenized and cached on Google Drive

In [ ]:
# ── Cell 0: Configuration ─────────────────────────────────────────
# Set this based on TinyStories ablation results:
V_THETA_VARIANT = 'gaussian'    # 'gaussian' or 'sarf'
K_MIX           = 8             # for 'gaussian' variant
SARF_N_ANCHORS  = 64            # for 'sarf' variant
W_SCALE         = 1.0
BG_QUAD_EPS     = 0.0           # set > 0 if G5 wins TinyStories

print(f'Phase 5 config: V_theta={V_THETA_VARIANT}')
if V_THETA_VARIANT == 'gaussian':
    print(f'  K_mix={K_MIX}, w_scale={W_SCALE}')
else:
    print(f'  SARF N_S={SARF_N_ANCHORS}, w_scale={W_SCALE}')
if BG_QUAD_EPS > 0:
    print(f'  background quadratic eps={BG_QUAD_EPS}')

In [ ]:
# ── Cell 1: Environment ───────────────────────────────────────────
import os, sys, gc, shutil, subprocess, json, time, math
from pathlib import Path
from dataclasses import asdict

os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')

REPO_URL    = 'https://github.com/dimitarpg13/semsimula-paper.git'
REPO_BRANCH = 'main'

IN_COLAB = 'google.colab' in sys.modules
print(f'IN_COLAB = {IN_COLAB}')


def _sh(cmd):
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f'exit {r.returncode}: {cmd}')


if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    REPO_ROOT = Path('/content/semsimula-paper')
    if not (REPO_ROOT / '.git').exists():
        if REPO_ROOT.exists():
            shutil.rmtree(REPO_ROOT)
        _sh(f'git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {REPO_ROOT}')
    else:
        try:
            _sh(f'git -C {REPO_ROOT} fetch --depth 1 origin {REPO_BRANCH}')
            _sh(f'git -C {REPO_ROOT} reset --hard origin/{REPO_BRANCH}')
        except RuntimeError as e:
            print(f'WARNING: repo refresh failed ({e}); using existing checkout.')

    GDRIVE_ROOT = Path('/content/drive/MyDrive/semsimula_fock_gaussian_sarf_openwebtext_phase5')
    GDRIVE_ROOT.mkdir(parents=True, exist_ok=True)

    DATA_DIR = GDRIVE_ROOT / 'data'
    DATA_DIR.mkdir(exist_ok=True)
    repo_data = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    if repo_data.is_symlink():
        repo_data.unlink()
    elif repo_data.is_dir():
        shutil.rmtree(repo_data)
    repo_data.symlink_to(DATA_DIR)

    CKPT_DIR    = GDRIVE_ROOT / 'checkpoints'
    RESULTS_DIR = GDRIVE_ROOT / 'results'
    CKPT_DIR.mkdir(exist_ok=True)
    RESULTS_DIR.mkdir(exist_ok=True)

    _sh('pip install -q transformers huggingface_hub pyarrow')
else:
    REPO_ROOT = Path('.').resolve()
    while not (REPO_ROOT / '.git').exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent
    DATA_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    CKPT_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'scaleup' / 'results' / 'phase5' / 'ckpts'
    RESULTS_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'scaleup' / 'results' / 'phase5'
    for d in [DATA_DIR, CKPT_DIR, RESULTS_DIR]:
        d.mkdir(parents=True, exist_ok=True)

CA_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch'
for sub in ['', 'parf', 'multixi', 'scaleup', 'sarf_mass_variant', 'energetic_minima']:
    d = str(CA_DIR / sub) if sub else str(CA_DIR)
    if d not in sys.path:
        sys.path.insert(0, d)

CKPT_PREFIX   = 'fock_gaussian_sarf_owt_phase5'
TOTAL_STEPS   = 200_000
CKPT_INTERVAL = 25_000
CKPT_STEPS    = list(range(CKPT_INTERVAL, TOTAL_STEPS + 1, CKPT_INTERVAL))

print(f'CKPT_DIR    = {CKPT_DIR}')
print(f'RESULTS_DIR = {RESULTS_DIR}')
print(f'Steps: {TOTAL_STEPS:,}  checkpoints at: {CKPT_STEPS}')

In [ ]:
# ── Cell 2: Resume logic ──────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}  ({props.total_memory/1e9:.1f} GB)')
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

resume_ckpt = None
resume_step = 0

existing = sorted(CKPT_DIR.glob(f'{CKPT_PREFIX}_*.pt'))
if existing:
    best_path = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'
    latest_step_path = existing[-1]

    if best_path.exists() and best_path.stat().st_mtime > latest_step_path.stat().st_mtime:
        resume_ckpt = best_path
    else:
        resume_ckpt = latest_step_path

    try:
        _meta = torch.load(resume_ckpt, map_location='cpu', weights_only=False)
        resume_step = _meta.get('step', 0)
        _ppl = _meta.get('val_ppl', float('nan'))
        print(f'Will resume from {resume_ckpt.name}  step={resume_step:,}  PPL={_ppl:.2f}')
        del _meta
    except Exception as e:
        print(f'[warn] {e}; starting from scratch')
        resume_ckpt = None
        resume_step = 0
else:
    print('No existing checkpoints — starting from step 0.')

In [ ]:
# ── Cell 3: Data loading (OpenWebText) ─────────────────────────────
from data_module import get_batch

MAX_TRAIN_TOKENS = 200_000_000
VAL_TOKENS       = 2_000_000
CHUNK_SIZE       = 50_000
VOCAB_SIZE       = 50257

from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('gpt2')

train_cache = DATA_DIR / f'openwebtext_train_{MAX_TRAIN_TOKENS // 1_000_000}M.npy'
val_cache   = DATA_DIR / f'openwebtext_val_{VAL_TOKENS // 1_000_000}M.npy'

if train_cache.exists() and val_cache.exists():
    print(f'Loading cached data ...')
    train_ids = np.load(train_cache)
    val_ids   = np.load(val_cache)
else:
    from datasets import load_dataset
    print('Downloading + tokenising OpenWebText (first run only) ...')
    ds = load_dataset('Skylion007/openwebtext', split='train', trust_remote_code=True)

    all_ids = []
    n_tok = 0
    target = MAX_TRAIN_TOKENS + VAL_TOKENS
    for i, row in enumerate(ds):
        ids = tok.encode(row['text'])
        all_ids.extend(ids)
        n_tok += len(ids)
        if i % CHUNK_SIZE == 0:
            print(f'  {n_tok:,} tokens from {i:,} docs ...', end='\r')
        if n_tok >= target:
            break

    all_ids = np.array(all_ids[:target], dtype=np.uint16)
    val_ids   = all_ids[:VAL_TOKENS]
    train_ids = all_ids[VAL_TOKENS:VAL_TOKENS + MAX_TRAIN_TOKENS]

    np.save(train_cache, train_ids)
    np.save(val_cache, val_ids)
    print(f'\nSaved: {train_cache}  ({len(train_ids):,} tokens)')
    print(f'       {val_cache}  ({len(val_ids):,} tokens)')

print(f'train: {len(train_ids):,}   val: {len(val_ids):,}')

In [ ]:
# ── Cell 4: Model config + Gaussian/SARF V_theta ──────────────────
import math
from model_fock_parf_multixi import FockMultiXiPARFLM, FockMultiXiPARFConfig
import model_fock_parf_v2
import model_parf_multixi
import model_parf
import model_parf_sparse

XI_CHANNELS    = 4
XI_ALPHA_INITS = [0.25, 0.50, 0.75, 0.95]
LAMBDA_V       = 1e-2
BLOCK_SIZE     = 512

LOGFREQ_PATH = CA_DIR / 'scaleup' / 'results' / 'logfreq_surprisal_openwebtext.npy'
DRIVE_LOGFREQ = RESULTS_DIR / 'logfreq_surprisal_openwebtext.npy'

if LOGFREQ_PATH.exists():
    LOGFREQ_FILE = LOGFREQ_PATH
elif DRIVE_LOGFREQ.exists():
    LOGFREQ_FILE = DRIVE_LOGFREQ
else:
    counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE).astype(np.float64)
    p = (counts + 1.0) / (counts.sum() + VOCAB_SIZE)
    surprisal = (-np.log(p)).astype(np.float32)
    LOGFREQ_FILE = DRIVE_LOGFREQ
    LOGFREQ_FILE.parent.mkdir(parents=True, exist_ok=True)
    np.save(LOGFREQ_FILE, surprisal)

print(f'Logfreq: {LOGFREQ_FILE}')

# Adaptive architecture tiers
ARCH_TIERS = [
    (384, 16, 32),
    (384, 12, 16),
    (256, 16, 16),
    (256,  8, 16),
]


def make_config(d, L, n_registers):
    return FockMultiXiPARFConfig(
        vocab_size=VOCAB_SIZE, d=d, max_len=1024,
        L=L, v_hidden=1024, v_depth=3, dt=1.0,
        mass_mode='logfreq',
        logfreq_path=str(LOGFREQ_FILE),
        logfreq_init_alpha=0.1,
        init_gamma=1.0,
        fixed_gamma=0.30,
        causal_force=True,
        ln_after_step=True,
        xi_channels=XI_CHANNELS,
        xi_alpha_inits=XI_ALPHA_INITS,
        xi_learnable=True,
        xi_alpha_init_mode='explicit',
        v_phi_kind='structural_competitive',
        v_phi_phi_hidden=128,
        v_phi_theta_hidden=128,
        top_k=8,
        score_head_hidden=32,
        gumbel_tau_init=1.0,
        gumbel_tau_min=0.3,
        gumbel_noise=True,
        use_gathered_v_phi=True,
        use_layer_checkpoint=True,
        ln_before_distance=True,
        per_layer_v_phi_scale=True,
        fock_version='v2',
        n_registers=n_registers,
        register_salience_decay=0.5,
        register_salience_threshold=0.005,
        creation_gate_hidden=64,
        stack_discipline=True,
        d_k=64,
        tau_create_init=8.0,
        reverse_channel=True,
        per_register_tau=True,
        per_register_keys=True,
        ortho_register_init=True,
    )


def build_gaussian_vtheta(model, d, variant, device):
    """Swap model.V_theta with Gaussian or SARF variant.

    Applies all stability fixes from TinyStories ablation:
      - Gaussian: init_log_precision=-log(d), precision_max=2/d
      - SARF: init_log_sigma=2.77, log_sigma_max=0.5*log(d)+1.0, anchor normalisation
    """
    xi_d = XI_CHANNELS * d
    from model_gaussian_vtheta import (
        MixtureGaussianVTheta, SARFGaussianVTheta, GaussianVThetaMultiXiAdapter,
    )

    if variant == 'gaussian':
        # init_log_precision = -log(d): ensures sigma_eff ≈ sqrt(d) from step 1,
        # matching the LN-normalised hidden-state scale (||h_L|| ≈ sqrt(d)).
        _init_log_prec = -math.log(d)
        # precision_max = 2/d: prevents a_k → ∞ (delta spikes, force divergence).
        _prec_max = 2.0 / d

        inner = MixtureGaussianVTheta(
            d=d, K=K_MIX, w_scale=W_SCALE, xi_d=xi_d,
            init_log_precision=_init_log_prec,
            precision_max=_prec_max,
        )
        model.V_theta = GaussianVThetaMultiXiAdapter(inner, K=XI_CHANNELS, d=d).to(device)

        _sigma_eff_init = 1.0 / (math.exp(_init_log_prec) ** 0.5)
        _sigma_min = 1.0 / (_prec_max ** 0.5)
        print(f'V_theta -> Gaussian(K={K_MIX}, w_scale={W_SCALE},'
              f' sigma_eff_init={_sigma_eff_init:.2f}, sigma_min={_sigma_min:.2f})')

    elif variant == 'sarf':
        # Compute SARF anchors from OWT PMI
        WINDOW = 5
        TOP_V = 8192
        token_counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE)
        top_v_ids = np.argsort(-token_counts)[:TOP_V]
        id_to_local = np.full(VOCAB_SIZE, -1, dtype=np.int64)
        id_to_local[top_v_ids] = np.arange(TOP_V)

        cooc = np.zeros((TOP_V, TOP_V), dtype=np.float64)
        local_ids = id_to_local[train_ids.astype(np.int64)]
        for offset in range(1, WINDOW + 1):
            a = local_ids[:-offset]
            b = local_ids[offset:]
            valid = (a >= 0) & (b >= 0)
            np.add.at(cooc, (a[valid], b[valid]), 1.0)
        cooc = cooc + cooc.T

        row_sums = cooc.sum(axis=1, keepdims=True)
        total = cooc.sum()
        expected = row_sums * row_sums.T / total
        with np.errstate(divide='ignore', invalid='ignore'):
            pmi = np.log(cooc / np.maximum(expected, 1e-12))
        pmi = np.nan_to_num(pmi, nan=0.0, posinf=0.0, neginf=-20.0)

        np.fill_diagonal(pmi, -np.inf)
        pmi_peaks = pmi.max(axis=1)
        anchor_local_ids = np.argsort(-pmi_peaks)[:SARF_N_ANCHORS]
        anchor_token_ids = top_v_ids[anchor_local_ids]
        anchor_positions = model.E.weight.data[anchor_token_ids].detach().clone()
        print(f'SARF anchors: {SARF_N_ANCHORS} PMI-peak tokens selected')
        print(f'  PMI peak range: [{pmi_peaks[anchor_local_ids[-1]]:.2f}, '
              f'{pmi_peaks[anchor_local_ids[0]]:.2f}]')

        # log_sigma_max = 0.5*log(d)+1.0: caps sigma at ≈e×sqrt(d), preventing
        # well deactivation (sigma drift → flat wells → v_reg collapse).
        _log_sigma_max = 0.5 * math.log(d) + 1.0
        # init_log_sigma = 2.77 → sigma ≈ 16 ≈ sqrt(d), matching LN-scale h_L.
        _init_log_sigma = math.log(d) / 2.0  # ln(sqrt(d)) ≈ 2.77 for d=256

        inner = SARFGaussianVTheta(
            d=d, anchor_positions=anchor_positions, xi_d=xi_d, w_scale=W_SCALE,
            init_log_sigma=_init_log_sigma,
            log_sigma_max=_log_sigma_max,
        )
        model.V_theta = GaussianVThetaMultiXiAdapter(inner, K=XI_CHANNELS, d=d).to(device)

        # Re-normalise anchors to match ln_after_step hidden-state scale.
        # ln_after_step=True normalises h to unit variance per dim after every
        # Verlet step → ||h_L|| ≈ sqrt(d).  Raw embedding anchors have norm ≈ 0.3.
        # Without normalisation, ||h_L - a_j|| ≈ sqrt(d) >> 0.3 → Gaussian bumps vanish.
        with torch.no_grad():
            a = model.V_theta.inner.anchors
            a = (a - a.mean(dim=-1, keepdim=True)) / (a.std(dim=-1, keepdim=True) + 1e-5)
            model.V_theta.inner.anchors.copy_(a)

        print(f'V_theta -> SARF Gaussian(N_S={SARF_N_ANCHORS})')
        print(f'  Anchors re-normalised: norm={a.norm(dim=-1).mean():.2f}'
              f'  (target ~{d**0.5:.1f})')
        print(f'  sigma_init={model.V_theta.inner.sigma.mean():.2f}'
              f'  sigma_max={math.exp(_log_sigma_max):.2f}')
    else:
        raise ValueError(f'Unknown V_theta variant: {variant}')


# ── Try architecture tiers ─────────────────────────────────────────
model = None
model_cfg = None
for d, L, M in ARCH_TIERS:
    try:
        cfg = make_config(d, L, M)
        mdl = FockMultiXiPARFLM(cfg).to(DEVICE)
        n_v_theta_mlp = sum(p.numel() for p in mdl.V_theta.parameters())
        build_gaussian_vtheta(mdl, d, V_THETA_VARIANT, DEVICE)
        n = mdl.num_params()
        n_v_theta = sum(p.numel() for p in mdl.V_theta.parameters())
        print(f'Trying d={d} L={L} M={M} -> {n:,} params '
              f'(V_theta {n_v_theta_mlp:,} MLP -> {n_v_theta:,} Gaussian)')
        if DEVICE == 'cuda':
            _rng = np.random.default_rng(42)
            _xb, _yb = get_batch(train_ids, 2, BLOCK_SIZE, _rng)
            _x = torch.from_numpy(_xb).to(DEVICE)
            _y = torch.from_numpy(_yb).to(DEVICE)
            _, _loss = mdl(_x, _y)
            _loss.backward()
            mdl.zero_grad(set_to_none=True)
            del _x, _y, _xb, _yb, _loss
            torch.cuda.empty_cache()
            print(f'OOM probe passed (batch=2)')
        model = mdl
        model_cfg = cfg
        break
    except RuntimeError as e:
        if 'out of memory' in str(e).lower():
            print(f'  OOM at d={d} L={L} M={M} — trying next tier ...')
            del mdl
            gc.collect()
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()
            continue
        raise

if model is None:
    raise RuntimeError('All architecture tiers OOMed.')

# ── Auto batch size ────────────────────────────────────────────────
BATCH_SIZE = 4
GRAD_ACCUM = 2
if DEVICE == 'cuda':
    for bs in [8, 6, 4]:
        try:
            _rng = np.random.default_rng(42)
            _xb, _yb = get_batch(train_ids, bs, BLOCK_SIZE, _rng)
            _x = torch.from_numpy(_xb).to(DEVICE)
            _y = torch.from_numpy(_yb).to(DEVICE)
            _, _loss = model(_x, _y)
            _loss.backward()
            model.zero_grad(set_to_none=True)
            del _x, _y, _xb, _yb, _loss
            torch.cuda.empty_cache()
            BATCH_SIZE = bs
            GRAD_ACCUM = max(1, 8 // bs)
            print(f'Auto batch: {bs} x accum={GRAD_ACCUM} (eff={bs*GRAD_ACCUM})')
            break
        except RuntimeError:
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()
            continue

EFFECTIVE_BATCH = BATCH_SIZE * GRAD_ACCUM
n_params = model.num_params()
n_v_theta = sum(p.numel() for p in model.V_theta.parameters())
IS_GAUSSIAN = V_THETA_VARIANT in ('gaussian', 'sarf')

print(f'\nModel: FockMultiXiPARFLM v2.1 + Gaussian V_theta (Phase 5)')
print(f'  params: {n_params:,}  (V_theta: {n_v_theta:,})')
print(f'  d={model_cfg.d}  L={model_cfg.L}  M={model_cfg.n_registers}')
print(f'  V_theta={V_THETA_VARIANT}  lambda_V={LAMBDA_V}')
print(f'  batch={BATCH_SIZE} x accum={GRAD_ACCUM} (eff={EFFECTIVE_BATCH})')
print(f'  IS_GAUSSIAN={IS_GAUSSIAN}')

In [ ]:
# ── Cell 5: Training loop ─────────────────────────────────────────
LR            = 2e-4      # higher than Phase 4 (bounded V tolerates it)
WEIGHT_DECAY  = 0.01
WARMUP_STEPS  = 8000
GRAD_CLIP     = 0.5       # relaxed vs Phase 4 (bounded forces)
EVAL_INTERVAL = 2000
EVAL_ITERS    = 40
LOG_INTERVAL  = 200
SEED          = 0

# Watchdog: relaxed because Gaussian V is bounded
GRAD_NORM_EMA_ALPHA = 0.05
GRAD_NORM_EMA_THRESHOLD = 20.0
GRAD_NORM_EMA_PATIENCE = 200

torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)


def lr_schedule(step):
    if step < WARMUP_STEPS:
        return LR * (step + 1) / WARMUP_STEPS
    progress = (step - WARMUP_STEPS) / max(TOTAL_STEPS - WARMUP_STEPS, 1)
    return LR * 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))


def forward_with_vreg(x, targets, lambda_v):
    h0 = model._embed(x)
    h_L, _ = model._stack_forward(h0, x, return_trajectory=False)
    logits = h_L @ model.E.weight.T
    loss_ntp = F.cross_entropy(
        logits.reshape(-1, model_cfg.vocab_size),
        targets.reshape(-1),
    )
    v_reg_value = torch.tensor(0.0, device=x.device)
    if lambda_v > 0:
        xis = model.xi_module(h_L.detach())
        V_vals = model.V_theta(xis, h_L)
        # Gaussian V is bounded: plain V^2 is safe (no log1p needed)
        v_reg_value = (V_vals ** 2).mean()
        if BG_QUAD_EPS > 0:
            bg = BG_QUAD_EPS * (h_L ** 2).sum(dim=-1, keepdim=True).mean()
            loss = loss_ntp + lambda_v * v_reg_value + bg
        else:
            loss = loss_ntp + lambda_v * v_reg_value
    else:
        loss = loss_ntp
    return loss, loss_ntp, v_reg_value


@torch.no_grad()
def evaluate():
    model.eval()
    losses = []
    for _ in range(EVAL_ITERS):
        xb, yb = get_batch(val_ids, BATCH_SIZE, BLOCK_SIZE, rng)
        x = torch.from_numpy(xb).to(DEVICE)
        y = torch.from_numpy(yb).to(DEVICE)
        with torch.enable_grad():
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))


def save_checkpoint(step_num, val_loss_val, tag_suffix=''):
    ckpt = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optim.state_dict(),
        'model_cfg': asdict(model_cfg),
        'train_cfg': {
            'batch_size': BATCH_SIZE, 'block_size': BLOCK_SIZE,
            'grad_accum': GRAD_ACCUM, 'effective_batch': EFFECTIVE_BATCH,
            'steps': TOTAL_STEPS, 'lr': LR, 'weight_decay': WEIGHT_DECAY,
            'warmup_steps': WARMUP_STEPS, 'grad_clip': GRAD_CLIP,
            'lambda_v': LAMBDA_V, 'v_theta_variant': V_THETA_VARIANT,
        },
        'step': step_num,
        'val_loss': val_loss_val,
        'val_ppl': math.exp(val_loss_val),
        'gamma': model.gamma.item(),
        'xi_alphas': model.xi_alpha_values(),
        'variant': f'fock_parf_multixi_v2.1_gaussian_{V_THETA_VARIANT}',
        'corpus': 'openwebtext',
        'phase': 5,
        'seed': SEED,
    }
    fname = f'{CKPT_PREFIX}_step{step_num}{tag_suffix}.pt'
    path = CKPT_DIR / fname
    torch.save(ckpt, path)
    print(f'  Checkpoint saved: {path}  (PPL={math.exp(val_loss_val):.2f})')
    return path


# ── Optimizer ──
optim = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=LR,
    weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95),
)

# ── Resume ──
if resume_ckpt is not None and resume_step < TOTAL_STEPS:
    print(f'Resuming from checkpoint at step {resume_step:,}: {resume_ckpt}')
    ckpt_data = torch.load(resume_ckpt, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt_data['model_state_dict'], strict=False)
    if 'optimizer_state_dict' in ckpt_data:
        try:
            optim.load_state_dict(ckpt_data['optimizer_state_dict'])
            print('  Optimizer state restored.')
        except (ValueError, KeyError) as e:
            print(f'  [info] Optimizer state incompatible, starting fresh: {e}')
    prev_ppl = ckpt_data.get('val_ppl', float('nan'))
    print(f'  Model loaded. Previous PPL: {prev_ppl:.2f}')
    del ckpt_data
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

# ── Training state ──
log_path = RESULTS_DIR / 'training_log.jsonl'
log_f = log_path.open('a')

t0 = time.time()
model.train()
run_ntp = 0.0
run_vreg = 0.0
n_run = 0
n_skipped = 0

best_val_ppl = float('inf')
_best_ckpt_path = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'
if _best_ckpt_path.exists():
    try:
        _bd = torch.load(_best_ckpt_path, map_location='cpu', weights_only=False)
        best_val_ppl = _bd.get('val_ppl', float('inf'))
        print(f'Restored running best PPL: {best_val_ppl:.2f}')
        del _bd
    except Exception as e:
        print(f'[warn] {e}')

# ── EMA watchdog (relaxed for Gaussian V) ──
_grad_norm_ema = 0.0
_grad_norm_above_thresh = 0

def _reload_best():
    if not _best_ckpt_path.exists():
        return resume_step
    ckpt = torch.load(_best_ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'], strict=False)
    try:
        optim.load_state_dict(ckpt['optimizer_state_dict'])
    except (ValueError, KeyError):
        pass
    s = ckpt.get('step', 0)
    p = ckpt.get('val_ppl', float('nan'))
    del ckpt
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    print(f'[watchdog] Reloaded best: step {s:,} PPL {p:.2f}')
    return s

steps_this_session = 0

print(f'\n{"="*60}')
print(f'Phase 5 (Gaussian {V_THETA_VARIANT}): steps {resume_step+1:,} -> {TOTAL_STEPS:,}')
print(f'  batch={BATCH_SIZE} x accum={GRAD_ACCUM} (eff={EFFECTIVE_BATCH})')
print(f'  block={BLOCK_SIZE}  lr={LR}  warmup={WARMUP_STEPS}  grad_clip={GRAD_CLIP}')
print(f'  d={model_cfg.d}  L={model_cfg.L}  M={model_cfg.n_registers}')
print(f'  V_theta={V_THETA_VARIANT}  params={n_params:,}')
print(f'  watchdog: threshold={GRAD_NORM_EMA_THRESHOLD} patience={GRAD_NORM_EMA_PATIENCE}')
print(f'{"="*60}\n')

for step in range(resume_step, TOTAL_STEPS):
    lr_now = lr_schedule(step)
    for g in optim.param_groups:
        g['lr'] = lr_now

    optim.zero_grad(set_to_none=True)
    accum_ntp = 0.0
    accum_vreg = 0.0
    for _acc in range(GRAD_ACCUM):
        xb, yb = get_batch(train_ids, BATCH_SIZE, BLOCK_SIZE, rng)
        x = torch.from_numpy(xb).to(DEVICE)
        y = torch.from_numpy(yb).to(DEVICE)
        loss, loss_ntp, v_reg = forward_with_vreg(x, y, LAMBDA_V)
        (loss / GRAD_ACCUM).backward()
        accum_ntp  += loss_ntp.item()       / GRAD_ACCUM
        accum_vreg += float(v_reg.detach()) / GRAD_ACCUM

    grad_norm = nn.utils.clip_grad_norm_(
        [p for p in model.parameters() if p.requires_grad], GRAD_CLIP,
    )
    if torch.isfinite(grad_norm) and math.isfinite(accum_ntp):
        optim.step()
        # Projected gradient: clamp log_sigma back into the feasible set so Adam's
        # momentum cannot push sigma past log_sigma_max even when the forward-pass
        # clamp makes the gradient surface flat beyond the boundary.
        if IS_GAUSSIAN and hasattr(getattr(model.V_theta, 'inner', None), 'clamp_params'):
            model.V_theta.inner.clamp_params()
    else:
        n_skipped += 1
        optim.zero_grad(set_to_none=True)

    # ── Watchdog ──
    _raw_gn = float(grad_norm)
    _grad_norm_ema = (1 - GRAD_NORM_EMA_ALPHA) * _grad_norm_ema + GRAD_NORM_EMA_ALPHA * _raw_gn
    if _grad_norm_ema > GRAD_NORM_EMA_THRESHOLD:
        _grad_norm_above_thresh += 1
    else:
        _grad_norm_above_thresh = 0

    if _grad_norm_above_thresh >= GRAD_NORM_EMA_PATIENCE:
        print(f'\n[watchdog] EMA grad_norm={_grad_norm_ema:.1f} > {GRAD_NORM_EMA_THRESHOLD} '
              f'for {_grad_norm_above_thresh} steps at step {step+1}.')
        _reload_best()
        _grad_norm_ema = 0.0
        _grad_norm_above_thresh = 0
        n_skipped += 1

    run_ntp += accum_ntp
    run_vreg += accum_vreg
    n_run += 1
    steps_this_session += 1

    if (step + 1) % LOG_INTERVAL == 0:
        avg_ntp = run_ntp / n_run
        avg_vreg = run_vreg / n_run
        run_ntp, run_vreg, n_run = 0.0, 0.0, 0
        elapsed = time.time() - t0
        sec_per_step = elapsed / steps_this_session
        remaining = (TOTAL_STEPS - step - 1) * sec_per_step
        alphas = model.xi_alpha_values()
        alpha_str = ','.join(f'{a:.3f}' for a in alphas)
        print(
            f'step {step+1:7d}/{TOTAL_STEPS}  '
            f'ntp={avg_ntp:.4f}  v_reg={avg_vreg:.4f}  lr={lr_now:.2e}  '
            f'grad={float(grad_norm):.2f}  gamma={model.gamma.item():.3f}  '
            f'alpha=[{alpha_str}]  '
            f'{elapsed:.0f}s  (~{remaining/3600:.1f}h remaining)'
        )
        log_f.write(json.dumps({
            'step': step + 1, 'train_loss': avg_ntp, 'v_reg': avg_vreg,
            'lr': lr_now, 'grad_norm': float(grad_norm),
            'gamma': model.gamma.item(), 'xi_alphas': alphas,
            'elapsed_sec': elapsed, 'sec_per_step': sec_per_step,
        }) + '\n')
        log_f.flush()

    if (step + 1) % EVAL_INTERVAL == 0:
        val_loss = evaluate()
        val_ppl = math.exp(val_loss)
        is_best = val_ppl < best_val_ppl
        if is_best:
            best_val_ppl = val_ppl
        elapsed = time.time() - t0
        marker = '*** NEW BEST ***' if is_best else ''
        print(f'>>> EVAL step {step+1:,}  val_loss={val_loss:.4f}  '
              f'val_ppl={val_ppl:.2f}  best={best_val_ppl:.2f}  '
              f'{marker}  ({elapsed:.0f}s)')
        log_f.write(json.dumps({
            'step': step + 1, 'val_loss': val_loss,
            'val_ppl': val_ppl, 'best_ppl': best_val_ppl,
        }) + '\n')
        log_f.flush()
        if is_best:
            save_checkpoint(step + 1, val_loss, tag_suffix='_best')

    if (step + 1) in set(CKPT_STEPS):
        if (step + 1) % EVAL_INTERVAL != 0:
            val_loss = evaluate()
            val_ppl = math.exp(val_loss)
        save_checkpoint(step + 1, val_loss)

log_f.close()
print(f'\nPhase 5 training complete. Best PPL: {best_val_ppl:.2f}')